<a href="https://colab.research.google.com/github/SukritiChauhan/Athletix/blob/main/05_vit_resnet_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
CKPT = "/content/drive/MyDrive/cifar_v2"
os.makedirs(CKPT, exist_ok=True)

if os.path.exists("/content/drive/MyDrive/cifar_data") and not os.path.exists("./data"):
    shutil.copytree("/content/drive/MyDrive/cifar_data", "./data")
    print("restored dataset from Drive")

mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(mean, std)])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

full_train = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
full_train_eval = torchvision.datasets.CIFAR10("./data", train=True, download=False, transform=test_tf)
test_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

g = torch.Generator().manual_seed(1234)
perm = torch.randperm(50000, generator=g).tolist()
train_idx, val_idx = perm[:45000], perm[45000:]
train_loader = DataLoader(Subset(full_train, train_idx), batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(full_train_eval, val_idx), batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)
print("train / val / test:", len(train_idx), len(val_idx), len(test_set))

class SmallCNN(nn.Module):
    def __init__(self, p=0.3):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 64), block(64, 128), block(128, 256))
        self.drop = nn.Dropout(p)
        self.fc = nn.Linear(256 * 4 * 4, 10)
    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.fc(self.drop(x))

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item(); total += y.size(0)
    return correct / total

def train(model, epochs=15, lr=0.05, aug_fn=None):
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=epochs,
                                                steps_per_epoch=len(train_loader))
    for ep in range(epochs):
        model.train(); running = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            if aug_fn is not None:
                x = aug_fn(x)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward(); opt.step(); sched.step()
            running += loss.item()
        print(f"  epoch {ep+1}: loss {running/len(train_loader):.3f}, val acc {evaluate(model, val_loader):.3f}")

def get_model(tag, seed, aug_fn=None):
    path = f"{CKPT}/{tag}_seed{seed}.pt"
    if os.path.exists(path):
        model = SmallCNN().to(device)
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"loaded {tag} seed {seed}")
    else:
        set_seed(seed)
        model = SmallCNN().to(device)
        print(f"training {tag} seed {seed}")
        train(model, aug_fn=aug_fn)
        torch.save(model.state_dict(), path)
    model.eval()
    return model

std_models = [get_model("std", s) for s in range(5)]
for s, m in enumerate(std_models):
    print(f"std seed {s}: val acc {evaluate(m, val_loader):.3f} | test acc {evaluate(m, test_loader):.3f}")

Mounted at /content/drive
device: cuda
restored dataset from Drive
train / val / test: 45000 5000 10000
loaded std seed 0
loaded std seed 1
loaded std seed 2
loaded std seed 3
loaded std seed 4
std seed 0: val acc 0.900 | test acc 0.902
std seed 1: val acc 0.902 | test acc 0.894
std seed 2: val acc 0.901 | test acc 0.897
std seed 3: val acc 0.899 | test acc 0.900
std seed 4: val acc 0.896 | test acc 0.900


In [2]:
val_y = torch.tensor([full_train_eval.targets[i] for i in val_idx])
test_y = torch.tensor(test_set.targets)

def ece_score(p, y, bins=15):
    conf, pred = p.max(1)
    correct = (pred == y).float()
    edges = torch.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.float().mean() * (correct[m].mean() - conf[m].mean()).abs()
    return float(e)

def nll_score(p, y):
    return float(-torch.log(p[torch.arange(len(y)), y].clamp_min(1e-12)).mean())

def metrics(p, y):
    return dict(acc=float((p.argmax(1) == y).float().mean()),
                conf=float(p.max(1)[0].mean()),
                ece=ece_score(p, y), nll=nll_score(p, y))

def apply_T(p, Tm):
    return F.softmax(torch.log(p.clamp_min(1e-12)) / Tm, 1)

def fit_T(p_val, y_val):
    best_nll, best_T = 1e9, 1.0
    for Tm in torch.arange(0.5, 3.01, 0.05):
        n = nll_score(apply_T(p_val, float(Tm)), y_val)
        if n < best_nll:
            best_nll, best_T = n, float(Tm)
    return best_T

In [3]:
import torchvision.transforms.functional as TF

mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
def to_pix(x): return x * std_t + mean_t
def to_norm(x): return (x - mean_t) / std_t

def add_noise(x, sigma):
    p = to_pix(x)
    return to_norm((p + torch.randn_like(p) * sigma).clamp(0, 1))

def blur(x, sigma):
    p = to_pix(x)
    return to_norm(TF.gaussian_blur(p, kernel_size=5, sigma=sigma).clamp(0, 1))

def noise_blur_aug(x):
    pick = torch.rand(x.size(0), device=device)
    out = x.clone()
    noise_m = pick < 0.25
    blur_m = (pick >= 0.25) & (pick < 0.5)
    if noise_m.any():
        s = torch.rand(int(noise_m.sum()), 1, 1, 1, device=device) * 0.23 + 0.02
        out[noise_m] = add_noise(x[noise_m], s)
    if blur_m.any():
        out[blur_m] = blur(x[blur_m], float(torch.empty(1).uniform_(0.5, 1.6)))
    return out

aug_models = [get_model("aug", s, aug_fn=noise_blur_aug) for s in range(5)]
for s, m in enumerate(aug_models):
    print(f"aug seed {s}: val acc {evaluate(m, val_loader):.3f} | test acc {evaluate(m, test_loader):.3f}")

loaded aug seed 0
loaded aug seed 1
loaded aug seed 2
loaded aug seed 3
loaded aug seed 4
aug seed 0: val acc 0.870 | test acc 0.874
aug seed 1: val acc 0.877 | test acc 0.873
aug seed 2: val acc 0.872 | test acc 0.871
aug seed 3: val acc 0.873 | test acc 0.870
aug seed 4: val acc 0.872 | test acc 0.868


In [4]:
os.system("pip -q install timm")
import timm

class Wrapped(nn.Module):
    def __init__(self, net, mean_in, std_in):
        super().__init__()
        self.net = net
        self.register_buffer("m_in", torch.tensor(mean_in).view(1, 3, 1, 1))
        self.register_buffer("s_in", torch.tensor(std_in).view(1, 3, 1, 1))
    def forward(self, x):
        p = x * std_t + mean_t
        p = F.interpolate(p, size=224, mode="bilinear", align_corners=False)
        return self.net((p - self.m_in) / self.s_in)

def make_pre(kind):
    if kind == "vit":
        net = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=10)
        return Wrapped(net, (0.5,) * 3, (0.5,) * 3).to(device)
    net = timm.create_model("resnet18", pretrained=True, num_classes=10)
    return Wrapped(net, (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)).to(device)

def finetune(model, kind, epochs=3):
    if kind == "vit":
        opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05); max_lr = 1e-4
    else:
        opt = torch.optim.SGD(model.parameters(), lr=0.02, momentum=0.9, nesterov=True, weight_decay=5e-4); max_lr = 0.02
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, total_steps=epochs * len(train_loader), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda")
    for ep in range(epochs):
        model.train(); running = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()
        print(f"  epoch {ep+1}: loss {running/len(train_loader):.3f}, val acc {evaluate(model, val_loader):.3f}")

def get_pre(kind, seed):
    path = f"{CKPT}/{kind}_seed{seed}.pt"
    set_seed(seed)
    model = make_pre(kind)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"loaded {kind} seed {seed}")
    else:
        print(f"training {kind} seed {seed}")
        finetune(model, kind)
        torch.save(model.state_dict(), path)
    model.eval()
    return model

vit_models = [get_pre("vit", s) for s in range(3)]
r18_models = [get_pre("r18", s) for s in range(3)]
for tag, group in [("vit", vit_models), ("r18", r18_models)]:
    for s, m in enumerate(group):
        print(f"{tag} seed {s}: val acc {evaluate(m, val_loader):.3f} | test acc {evaluate(m, test_loader):.3f}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

loaded vit seed 0


loaded vit seed 1
loaded vit seed 2


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

loaded r18 seed 0
training r18 seed 1
  epoch 1: loss 1.025, val acc 0.821
  epoch 2: loss 0.346, val acc 0.900
  epoch 3: loss 0.278, val acc 0.910
training r18 seed 2
  epoch 1: loss 1.087, val acc 0.867
  epoch 2: loss 0.359, val acc 0.902
  epoch 3: loss 0.278, val acc 0.914
vit seed 0: val acc 0.973 | test acc 0.971
vit seed 1: val acc 0.974 | test acc 0.971
vit seed 2: val acc 0.971 | test acc 0.972
r18 seed 0: val acc 0.918 | test acc 0.914
r18 seed 1: val acc 0.910 | test acc 0.915
r18 seed 2: val acc 0.914 | test acc 0.913


In [5]:
os.system("pip -q install datasets")
from sklearn.metrics import roc_auc_score
from datasets import load_dataset

X_test = torch.cat([x for x, _ in test_loader]).to(device)
X_val = torch.cat([x for x, _ in val_loader]).to(device)

def probs_X(model, X, bs=250):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(F.softmax(model(X[i:i + bs]).float(), 1).cpu())
    return torch.cat(out)

def contrast(x, f):
    p = to_pix(x); m = p.mean(dim=(1, 2, 3), keepdim=True)
    return to_norm(((p - m) * f + m).clamp(0, 1))
def brightness(x, d):
    return to_norm((to_pix(x) + d).clamp(0, 1))

conds = [("clean", lambda x: x),
         ("noise 0.05", lambda x: add_noise(x, 0.05)), ("noise 0.1", lambda x: add_noise(x, 0.1)),
         ("noise 0.2", lambda x: add_noise(x, 0.2)),
         ("blur 0.7", lambda x: blur(x, 0.7)), ("blur 1.0", lambda x: blur(x, 1.0)), ("blur 1.5", lambda x: blur(x, 1.5)),
         ("contrast 0.5", lambda x: contrast(x, 0.5)), ("contrast 0.3", lambda x: contrast(x, 0.3)),
         ("contrast 0.15", lambda x: contrast(x, 0.15)), ("brightness +0.3", lambda x: brightness(x, 0.3))]

ds = load_dataset("ufldl-stanford/svhn", "cropped_digits", split="test[:10000]")
X_ood = torch.stack([test_tf(img.convert("RGB")) for img in ds["image"]]).to(device)

def auroc(p_in, p_out):
    s = torch.cat([p_in.max(1)[0], p_out.max(1)[0]]).numpy()
    lab = np.concatenate([np.ones(len(p_in)), np.zeros(len(p_out))])
    return roc_auc_score(lab, s)

def run_family(models):
    Ts = [fit_T(probs_X(m, X_val), val_y) for m in models]
    out, raw, P_clean = {}, None, None
    for cname, fn in conds:
        torch.manual_seed(123)
        P = [probs_X(m, fn(X_test)) for m in models]
        out[cname] = [metrics(apply_T(p, t), test_y) for p, t in zip(P, Ts)]
        if cname == "clean":
            raw, P_clean = [metrics(p, test_y) for p in P], P
    ood = [auroc(pc, probs_X(m, X_ood)) for pc, m in zip(P_clean, models)]
    return Ts, out, raw, ood

families = {"CNN scratch": std_models, "ResNet18 pre": r18_models, "ViT-tiny pre": vit_models}
R = {name: run_family(group) for name, group in families.items()}

def mstd(vals):
    v = torch.tensor(vals, dtype=torch.float32)
    return f"{v.mean():.3f}±{v.std():.3f}" if len(vals) > 1 else f"{v.mean():.3f}"

print("\nCLEAN (temperature fitted on validation)")
print(f"{'family':<14}{'acc':>14}{'ECE raw':>14}{'ECE +temp':>14}{'NLL +temp':>14}")
for name, (Ts, out, raw, ood) in R.items():
    print(f"{name:<14}{mstd([m['acc'] for m in out['clean']]):>14}{mstd([m['ece'] for m in raw]):>14}"
          f"{mstd([m['ece'] for m in out['clean']]):>14}{mstd([m['nll'] for m in out['clean']]):>14}")

print("\nSHIFT (acc / ECE, temperature from clean validation)")
print(f"{'condition':<18}" + "".join(f"{n:>22}" for n in families))
for cname, _ in conds:
    row = ""
    for name, (Ts, out, raw, ood) in R.items():
        a = sum(m['acc'] for m in out[cname]) / len(out[cname]); e = sum(m['ece'] for m in out[cname]) / len(out[cname])
        row += f"{a:.3f} / {e:.3f}".rjust(22)
    print(f"{cname:<18}{row}")

print("\nOOD detection vs SVHN (AUROC)")
for name, (Ts, out, raw, ood) in R.items():
    print(f"{name:<14}{mstd(ood)}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

cropped_digits/train-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  136MB            

cropped_digits/train-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

cropped_digits/test-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 47.0MB            

cropped_digits/test-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

cropped_digits/extra-00000-of-00002.parq(…): reconstructing file:   0%|          |  0.00B /  511MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/73257 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26032 [00:00<?, ? examples/s]

Generating extra split:   0%|          | 0/531131 [00:00<?, ? examples/s]


CLEAN (temperature fitted on validation)
family                   acc       ECE raw     ECE +temp     NLL +temp
CNN scratch      0.899±0.003   0.029±0.003   0.007±0.001   0.296±0.006
ResNet18 pre     0.914±0.001   0.013±0.002   0.006±0.001   0.250±0.003
ViT-tiny pre     0.971±0.001   0.004±0.001   0.004±0.001   0.084±0.002

SHIFT (acc / ECE, temperature from clean validation)
condition                    CNN scratch          ResNet18 pre          ViT-tiny pre
clean                      0.899 / 0.007         0.914 / 0.006         0.971 / 0.004
noise 0.05                 0.645 / 0.127         0.536 / 0.152         0.836 / 0.064
noise 0.1                  0.305 / 0.386         0.175 / 0.349         0.410 / 0.347
noise 0.2                  0.139 / 0.536         0.111 / 0.349         0.156 / 0.460
blur 0.7                   0.704 / 0.045         0.722 / 0.069         0.931 / 0.015
blur 1.0                   0.406 / 0.209         0.415 / 0.242         0.830 / 0.053
blur 1.5                 

In [6]:
print(vit_models[0].net.pretrained_cfg)
print(r18_models[0].net.pretrained_cfg)

def finetune_vit_aug(model, epochs=3, aug_fn=None):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-4, total_steps=epochs * len(train_loader), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda")
    for ep in range(epochs):
        model.train(); running = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            if aug_fn is not None:
                x = aug_fn(x)
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            running += loss.item()
        print(f"  epoch {ep+1}: loss {running/len(train_loader):.3f}, val acc {evaluate(model, val_loader):.3f}")

def get_vit_aug(seed):
    path = f"{CKPT}/vitaug_seed{seed}.pt"
    set_seed(seed)
    model = make_pre("vit")
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"loaded vitaug seed {seed}")
    else:
        print(f"training vitaug seed {seed}")
        finetune_vit_aug(model, aug_fn=noise_blur_aug)
        torch.save(model.state_dict(), path)
    model.eval()
    return model

vitaug_models = [get_vit_aug(s) for s in range(3)]
for s, m in enumerate(vitaug_models):
    print(f"vitaug seed {s}: val acc {evaluate(m, val_loader):.3f} | test acc {evaluate(m, test_loader):.3f}")

R2 = {"ViT-tiny": R["ViT-tiny pre"], "ViT-tiny + aug": run_family(vitaug_models)}

print("\nCLEAN")
print(f"{'family':<16}{'acc':>14}{'ECE raw':>14}{'ECE +temp':>14}{'NLL +temp':>14}")
for name, (Ts, out, raw, ood) in R2.items():
    print(f"{name:<16}{mstd([m['acc'] for m in out['clean']]):>14}{mstd([m['ece'] for m in raw]):>14}"
          f"{mstd([m['ece'] for m in out['clean']]):>14}{mstd([m['nll'] for m in out['clean']]):>14}")
print("\nSHIFT (acc / ECE)")
print(f"{'condition':<18}" + "".join(f"{n:>22}" for n in R2))
for cname, _ in conds:
    row = ""
    for name, (Ts, out, raw, ood) in R2.items():
        a = sum(m['acc'] for m in out[cname]) / len(out[cname]); e = sum(m['ece'] for m in out[cname]) / len(out[cname])
        row += f"{a:.3f} / {e:.3f}".rjust(22)
    print(f"{cname:<18}{row}")
print("\nOOD detection vs SVHN (AUROC)")
for name, (Ts, out, raw, ood) in R2.items():
    print(f"{name:<16}{mstd(ood)}")

{'url': 'https://storage.googleapis.com/vit_models/augreg/Ti_16-i21k-300ep-lr_0.001-aug_none-wd_0.03-do_0.0-sd_0.0--imagenet2012-steps_20k-lr_0.03-res_224.npz', 'hf_hub_id': 'timm/vit_tiny_patch16_224.augreg_in21k_ft_in1k', 'architecture': 'vit_tiny_patch16_224', 'tag': 'augreg_in21k_ft_in1k', 'custom_load': True, 'input_size': (3, 224, 224), 'fixed_input_size': True, 'interpolation': 'bicubic', 'crop_pct': 0.9, 'crop_mode': 'center', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'num_classes': 1000, 'pool_size': None, 'first_conv': 'patch_embed.proj', 'classifier': 'head', 'license': 'apache-2.0'}
{'url': 'https://github.com/huggingface/pytorch-image-models/releases/download/v0.1-rsb-weights/resnet18_a1_0-d63eafa0.pth', 'hf_hub_id': 'timm/resnet18.a1_in1k', 'architecture': 'resnet18', 'tag': 'a1_in1k', 'custom_load': False, 'input_size': (3, 224, 224), 'test_input_size': (3, 288, 288), 'fixed_input_size': False, 'interpolation': 'bicubic', 'crop_pct': 0.95, 'test_crop_pct': 1.0, 'c

/tmp/ipykernel_1939/84690333.py:17: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  epoch 1: loss 0.827, val acc 0.951
  epoch 2: loss 0.290, val acc 0.966
  epoch 3: loss 0.207, val acc 0.971
training vitaug seed 1
  epoch 1: loss 0.801, val acc 0.954
  epoch 2: loss 0.288, val acc 0.966
  epoch 3: loss 0.209, val acc 0.973
training vitaug seed 2
  epoch 1: loss 0.735, val acc 0.951
  epoch 2: loss 0.288, val acc 0.969
  epoch 3: loss 0.211, val acc 0.971
vitaug seed 0: val acc 0.971 | test acc 0.968
vitaug seed 1: val acc 0.973 | test acc 0.971
vitaug seed 2: val acc 0.971 | test acc 0.968

CLEAN
family                     acc       ECE raw     ECE +temp     NLL +temp
ViT-tiny           0.971±0.001   0.004±0.001   0.004±0.001   0.084±0.002
ViT-tiny + aug     0.969±0.002   0.003±0.001   0.004±0.002   0.092±0.006

SHIFT (acc / ECE)
condition                       ViT-tiny        ViT-tiny + aug
clean                      0.971 / 0.004         0.969 / 0.004
noise 0.05                 0.836 / 0.064         0.932 / 0.010
noise 0.1                  0.410 / 0.347         